# Wind Turbine Blade Damage Detection — Improved YOLOv8m Baseline

**Improvements over the stock YOLOv8m baseline:**
1. **CBAM attention** (Convolutional Block Attention Module) added after SPPF and PAN neck C2f blocks
2. **Cosine LR schedule** with extended warmup
3. **Stronger augmentation**: MixUp + Copy-Paste
4. **Label smoothing** (ε=0.05)
5. **Longer training** (150 epochs with early stopping)
6. **TTA evaluation** (test-time augmentation)

Reference: Woo et al., CBAM: Convolutional Block Attention Module, ECCV 2018.

In [ ]:
# ============================================================
# 0) Environment & dependency setup
# ============================================================
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip -q uninstall -y wandb ray >/dev/null 2>&1
!pip -q install ultralytics==8.2.84 transformers==4.48.0 accelerate==0.33.0 timm==0.9.16 torchmetrics==1.4.0

import random, shutil, json, time
from pathlib import Path
import numpy as np, pandas as pd
from tqdm import tqdm
import torch
from ultralytics import YOLO

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Toggles
USE_VLM    = True   # BLIP scene captions
USE_LLM    = True   # FLAN-T5 inspection summaries
USE_CBAM   = True   # improved YOLOv8m-Wind with CBAM (False = stock baseline)
RUN_EVOLVE = False  # hyperparam evolution

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ============================================================
# 1) CBAM Module Definition & Registration
#    Convolutional Block Attention Module (Woo et al., ECCV 2018)
#    Inserted after SPPF + PAN neck C2f blocks for improved
#    small-defect detection on turbine blade surfaces.
# ============================================================
import torch
import torch.nn as nn


class ChannelAttention(nn.Module):
    """Channel-wise attention: recalibrate inter-channel relationships."""
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        mid = max(1, in_channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Linear(in_channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c = x.shape[:2]
        avg = self.avg_pool(x).view(b, c)
        mx  = self.max_pool(x).view(b, c)
        attn = self.sigmoid(self.shared_mlp(avg) + self.shared_mlp(mx))
        return x * attn.view(b, c, 1, 1)


class SpatialAttention(nn.Module):
    """Spatial attention: highlight defect-rich spatial locations."""
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_map, max_map], dim=1)))


class CBAM(nn.Module):
    """CBAM = ChannelAttention -> SpatialAttention (no channel change)."""
    def __init__(self, in_channels: int, reduction: int = 16, kernel_size: int = 7):
        super().__init__()
        self.channel_attn = ChannelAttention(in_channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))


# Register in ultralytics namespace so the YAML model can resolve "CBAM"
import ultralytics.nn.modules as _unn
_unn.CBAM             = CBAM
_unn.ChannelAttention = ChannelAttention
_unn.SpatialAttention = SpatialAttention
print("CBAM registered in ultralytics.nn.modules ✓")

In [ ]:
# ============================================================
# 2) Write custom model YAML (YOLOv8m-Wind with CBAM)
# ============================================================
YOLO_WIND_YAML = """
# YOLOv8m-Wind: YOLOv8m + CBAM attention for wind turbine blade damage detection
nc: 2
scales:
  m: [0.67, 0.75, 768]

backbone:
  - [-1, 1, Conv,  [64, 3, 2]]      # 0  P1/2
  - [-1, 1, Conv,  [128, 3, 2]]     # 1  P2/4
  - [-1, 3, C2f,   [128, True]]     # 2
  - [-1, 1, Conv,  [256, 3, 2]]     # 3  P3/8
  - [-1, 6, C2f,   [256, True]]     # 4  <- P3 skip
  - [-1, 1, Conv,  [512, 3, 2]]     # 5  P4/16
  - [-1, 6, C2f,   [512, True]]     # 6  <- P4 skip
  - [-1, 1, Conv,  [1024, 3, 2]]    # 7  P5/32
  - [-1, 3, C2f,   [1024, True]]    # 8
  - [-1, 1, SPPF,  [1024, 5]]       # 9
  - [-1, 1, CBAM,  [1024]]          # 10  CBAM on deepest backbone feature

head:
  - [-1,  1, nn.Upsample, [None, 2, 'nearest']]  # 11
  - [[-1, 6], 1, Concat,  [1]]                    # 12  cat P4
  - [-1,  3, C2f,   [512]]                        # 13
  - [-1,  1, CBAM,  [512]]                        # 14  CBAM on P4 neck feature

  - [-1,  1, nn.Upsample, [None, 2, 'nearest']]  # 15
  - [[-1, 4], 1, Concat,  [1]]                    # 16  cat P3
  - [-1,  3, C2f,   [256]]                        # 17  P3/8 small
  - [-1,  1, CBAM,  [256]]                        # 18  CBAM on P3 (small defects)

  - [-1,  1, Conv,  [256, 3, 2]]                  # 19
  - [[-1, 14], 1, Concat, [1]]                    # 20
  - [-1,  3, C2f,   [512]]                        # 21  P4/16 medium

  - [-1,  1, Conv,  [512, 3, 2]]                  # 22
  - [[-1, 10], 1, Concat, [1]]                    # 23
  - [-1,  3, C2f,   [1024]]                       # 24  P5/32 large

  - [[18, 21, 24], 1, Detect, [nc]]               # Detect(P3, P4, P5)
""".strip()

WORKDIR = Path("/kaggle/working")
PROJECT = "windturbine_yolo_improved"
OUT = WORKDIR / PROJECT
OUT.mkdir(parents=True, exist_ok=True)

model_yaml_path = OUT / "yolov8m_wind.yaml"
model_yaml_path.write_text(YOLO_WIND_YAML)
print(f"Model YAML written → {model_yaml_path}")

In [ ]:
# ============================================================
# 3) Dataset preparation
# ============================================================
ROOT       = Path("/kaggle/input/yolo-annotated-wind-turbines-586x371")
DS         = ROOT / "NordTank586x371"
IMG_DIR_IN = DS / "images"
LBL_DIR_IN = DS / "labels"
assert IMG_DIR_IN.exists() and LBL_DIR_IN.exists(), "Dataset paths not found."

# Create working directories
for split in ("train", "val", "test"):
    (OUT / f"dataset/images/{split}").mkdir(parents=True, exist_ok=True)
    (OUT / f"dataset/labels/{split}").mkdir(parents=True, exist_ok=True)
(OUT / "artifacts").mkdir(parents=True, exist_ok=True)

# --- Index images ---
images = sorted([*IMG_DIR_IN.glob("*.png"), *IMG_DIR_IN.glob("*.jpg")])

def label_status(stem):
    lb = LBL_DIR_IN / f"{stem}.txt"
    if not lb.exists(): return None, False
    txt = lb.read_text().strip()
    return lb, bool(txt)

pairs = []
for im in images:
    lb, has_obj = label_status(im.stem)
    pairs.append({"img": im, "lbl_in": lb, "has_obj": has_obj})

# --- Stratified split ---
def split3(items, train_ratio=0.8, val_ratio=0.1):
    items = items.copy(); random.Random(SEED).shuffle(items)
    n = len(items); n_tr = int(n * train_ratio); n_va = int(n * val_ratio)
    return items[:n_tr], items[n_tr:n_tr+n_va], items[n_tr+n_va:]

pos = [p for p in pairs if p["has_obj"]]
neg = [p for p in pairs if not p["has_obj"]]
tr_p,va_p,te_p = split3(pos)
tr_n,va_n,te_n = split3(neg)
train_set, val_set, test_set = tr_p+tr_n, va_p+va_n, te_p+te_n
print(f"Train:{len(train_set)} Val:{len(val_set)} Test:{len(test_set)}")

# --- Copy + create labels ---
def copy_split(split_name, split_list):
    n_lbl, n_empty = 0, 0
    for rec in split_list:
        im, lb_in = rec["img"], rec["lbl_in"]
        shutil.copy2(im, OUT/f"dataset/images/{split_name}/{im.name}")
        dst_lbl = OUT/f"dataset/labels/{split_name}/{im.stem}.txt"
        if lb_in is not None:
            txt = lb_in.read_text().strip()
            if txt: shutil.copy2(lb_in, dst_lbl); n_lbl += 1
            else:   dst_lbl.write_text(""); n_empty += 1
        else:       dst_lbl.write_text(""); n_empty += 1
    print(f"[{split_name}] labelled:{n_lbl} background:{n_empty}")

for name, lst in [("train",train_set),("val",val_set),("test",test_set)]:
    copy_split(name, lst)

# --- Sanitise labels ---
def sanitize(lbl_dir, nc=2):
    kept = dropped = 0
    for f in lbl_dir.glob("*.txt"):
        lines_out = []
        for ln in f.read_text().strip().splitlines():
            p = ln.strip().split()
            if len(p) != 5: dropped+=1; continue
            try: cid=int(float(p[0])); x,y,w,h=map(float,p[1:])
            except: dropped+=1; continue
            if not (0<=cid<nc) or w<=0 or h<=0: dropped+=1; continue
            x,y,w,h = [min(max(v,0.0),1.0) for v in (x,y,w,h)]
            lines_out.append(f"{cid} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
            kept += 1
        f.write_text("\n".join(lines_out))
    print(f"  Sanitized {lbl_dir.parent.name}/{lbl_dir.name}: kept={kept} dropped={dropped}")

for split in ("train","val","test"):
    sanitize(OUT/f"dataset/labels/{split}")

# --- data.yaml ---
DATA_YAML = OUT / "data.yaml"
DATA_YAML.write_text(f"""
path: {OUT}/dataset
train: images/train
val:   images/val
test:  images/test
nc: 2
names:
  0: dirt
  1: damage
""".strip())
print(f"data.yaml → {DATA_YAML}")

In [ ]:
# ============================================================
# 4) Build & train YOLOv8m-Wind (CBAM improved baseline)
# ============================================================
EPOCHS = 150
IMGSZ  = 640
BATCH  = 16
RUN_NAME = "yolov8m_wind_cbam" if USE_CBAM else "yolov8m_baseline"

if USE_CBAM:
    # Load custom YAML then transfer pretrained YOLOv8m backbone weights
    print("[Model] Loading YOLOv8m-Wind (CBAM) …")
    model = YOLO(str(model_yaml_path))

    pretrained = YOLO("yolov8m.pt")
    src_state = pretrained.model.state_dict()
    dst_state = model.model.state_dict()
    new_state = {}
    matched = skipped = 0
    for k, v in dst_state.items():
        if k in src_state and src_state[k].shape == v.shape:
            new_state[k] = src_state[k]; matched += 1
        else:
            new_state[k] = v; skipped += 1
    model.model.load_state_dict(new_state, strict=False)
    print(f"[Model] Weights transferred: matched={matched} skipped(CBAM)={skipped}")
else:
    print("[Model] Loading stock YOLOv8m baseline …")
    model = YOLO("yolov8m.pt")

device = 0 if torch.cuda.is_available() else "cpu"

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    project=str(OUT),
    name=RUN_NAME,
    seed=SEED,
    patience=30,
    cos_lr=True,            # cosine LR schedule
    warmup_epochs=5,        # extended warmup for CBAM
    close_mosaic=10,        # disable mosaic last 10 epochs
    mixup=0.15,             # MixUp augmentation
    copy_paste=0.10,        # Copy-Paste augmentation
    label_smoothing=0.05,   # label smoothing
    amp=True,               # mixed precision
)

BEST_CKPT = OUT / RUN_NAME / "weights" / "best.pt"
print(f"Best checkpoint → {BEST_CKPT}")

In [ ]:
# ============================================================
# 5) Evaluation: val + test (standard & TTA)
# ============================================================
model = YOLO(str(BEST_CKPT))

val_std  = model.val(split="val",  data=str(DATA_YAML), imgsz=IMGSZ,
                     save_json=True, plots=True, project=str(OUT), name="val_std")
test_std = model.val(split="test", data=str(DATA_YAML), imgsz=IMGSZ,
                     save_json=True, plots=True, project=str(OUT), name="test_std")
test_tta = model.val(split="test", data=str(DATA_YAML), imgsz=IMGSZ,
                     augment=True, plots=False, project=str(OUT), name="test_tta")

def mrow(tag, m):
    return {"tag": tag, "mAP50_95": float(m.box.map), "mAP50": float(m.box.map50),
            "mAP75": float(m.box.map75), "precision": float(m.box.mp), "recall": float(m.box.mr)}

metrics_df = pd.DataFrame([mrow("val_std",val_std), mrow("test_std",test_std), mrow("test_tta",test_tta)])
metrics_df.to_csv(OUT/"artifacts/summary_metrics.csv", index=False)

# Per-class AP
try:
    per_cls = getattr(test_std.box, "maps", None)
    if per_cls is not None and len(per_cls) == 2:
        pd.DataFrame({"class":["dirt","damage"], "AP50_95": list(per_cls)}).to_csv(
            OUT/"artifacts/per_class_ap.csv", index=False)
except Exception as e:
    print("Per-class AP:", e)

print(metrics_df.to_string(index=False))

In [ ]:
# ============================================================
# 6) Severity scoring & executive summary
# ============================================================
names = ["dirt", "damage"]
H, W  = 371, 586  # native dataset resolution

def detect_image(im_path, conf=0.25):
    res = model.predict(source=str(im_path), imgsz=IMGSZ, conf=conf, verbose=False)
    dets = []
    for r in res:
        if r.boxes is None: continue
        for b in r.boxes:
            cls_id = int(b.cls.item())
            dets.append({"cls": names[cls_id], "conf": float(b.conf.item()),
                         "bbox_xyxy": b.xyxy.cpu().numpy().tolist()[0]})
    return dets

test_imgs = sorted((OUT/"dataset/images/test").glob("*.*"))
records = []
for im in tqdm(test_imgs, desc="Scoring"):
    dets = detect_image(im)
    sev = sum(
        (2.0 if d["cls"]=="damage" else 1.0)
        * max(0, d["bbox_xyxy"][2]-d["bbox_xyxy"][0])
        * max(0, d["bbox_xyxy"][3]-d["bbox_xyxy"][1]) / (W*H)
        * d["conf"]
        for d in dets
    )
    risk = "high" if sev>0.05 else ("moderate" if sev>=0.01 else "low")
    records.append({"image": im.name,
                    "num_dirt":       sum(1 for d in dets if d["cls"]=="dirt"),
                    "num_damage":     sum(1 for d in dets if d["cls"]=="damage"),
                    "severity_score": round(sev, 6),
                    "risk_level":     risk,
                    "detections_json": json.dumps(dets)})

det_df = pd.DataFrame(records)
det_df.to_csv(OUT/"artifacts/detections_test.csv", index=False)

risk_counts = det_df["risk_level"].value_counts().to_dict()
summary = {
    "images_tested":      len(det_df),
    "images_with_damage": int((det_df["num_damage"]>0).sum()),
    "risk_high":          risk_counts.get("high",0),
    "risk_moderate":      risk_counts.get("moderate",0),
    "risk_low":           risk_counts.get("low",0),
    "mean_severity":      float(det_df["severity_score"].mean()),
    "p95_severity":       float(det_df["severity_score"].quantile(0.95)),
}
pd.DataFrame([summary]).to_csv(OUT/"artifacts/executive_summary.csv", index=False)
print("Executive summary:", summary)

In [ ]:
# ============================================================
# 7) Inference FPS benchmark
# ============================================================
subset = test_imgs[:200] if len(test_imgs)>200 else test_imgs
t0 = time.time()
for im in subset:
    model.predict(source=str(im), imgsz=IMGSZ, conf=0.25, verbose=False)
fps = len(subset)/(time.time()-t0)
pd.DataFrame([{"fps": fps, "images": len(subset), "imgsz": IMGSZ}]).to_csv(
    OUT/"artifacts/inference_fps.csv", index=False)
print(f"Inference FPS: {fps:.2f} on {len(subset)} images")

In [ ]:
# ============================================================
# 8) Optional: BLIP scene captions + FLAN-T5 inspection reports
# ============================================================
scene_caps = {}
if USE_VLM:
    from transformers import BlipProcessor, BlipForConditionalGeneration
    from PIL import Image
    blip_proc  = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
    blip_model.to("cuda" if torch.cuda.is_available() else "cpu")

    for im in tqdm(test_imgs[:400], desc="BLIP captions"):
        try:
            img  = Image.open(im).convert("RGB")
            inp  = blip_proc(images=img, return_tensors="pt").to(blip_model.device)
            out  = blip_model.generate(**inp, max_new_tokens=30)
            scene_caps[im.name] = blip_proc.decode(out[0], skip_special_tokens=True)
        except Exception:
            scene_caps[im.name] = ""
    with open(OUT/"artifacts/scene_captions.json","w") as f:
        json.dump(scene_caps, f, indent=2)

reports_with, reports_wo = [], []
if USE_LLM:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
    llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    llm.to("cuda" if torch.cuda.is_available() else "cpu")

    merged = det_df.copy()
    if USE_VLM: merged["scene_caption"] = merged["image"].map(scene_caps).fillna("")

    def gen_report(scene, dmg, dirt, sev):
        prompt = (
            "You are a UAV inspection engineer. Write a concise professional report "
            "(2-4 sentences) for a wind-turbine surface image.\n"
            f"Scene: {scene or 'UAV close-up of turbine blade/surface'}\n"
            f"Detections: damage={dmg}, dirt={dirt}\n"
            f"Severity score (0-1+): {sev:.3f}\n"
            "Rules: If damage>0, recommend a maintenance action. "
            "State risk level: low (<0.01), moderate (0.01-0.05), high (>0.05)."
        )
        inp = tok(prompt, return_tensors="pt").to(llm.device)
        out = llm.generate(**inp, max_new_tokens=120)
        return tok.decode(out[0], skip_special_tokens=True)

    subset_rows = merged.sample(min(150,len(merged)), random_state=SEED)
    for _, r in tqdm(subset_rows.iterrows(), total=len(subset_rows), desc="Reports (with BLIP)"):
        txt = gen_report(r.get("scene_caption",""), int(r["num_damage"]),
                         int(r["num_dirt"]), float(r["severity_score"]))
        reports_with.append({"image": r["image"], "inspection_report": txt,
                              "risk_level": r["risk_level"]})

    for _, r in tqdm(subset_rows.iterrows(), total=len(subset_rows), desc="Reports (no caption)"):
        txt = gen_report("", int(r["num_damage"]), int(r["num_dirt"]), float(r["severity_score"]))
        reports_wo.append({"image": r["image"], "inspection_report": txt,
                           "risk_level": r["risk_level"]})

    pd.DataFrame(reports_with).to_csv(OUT/"artifacts/inspection_reports_with_blip.csv", index=False)
    pd.DataFrame(reports_wo).to_csv(OUT/"artifacts/inspection_reports_without_blip.csv", index=False)
    print(f"Reports saved: {len(reports_with)} (with BLIP) | {len(reports_wo)} (without)")

In [ ]:
# ============================================================
# 9) Print artifact paths
# ============================================================
CITATIONS = """
Foster, Ashley et al. (2022), Drone Footage Wind Turbine Surface Damage Detection,
IEEE IVMSP 2022.
SHIHAVUDDIN, A.S.M.; Chen, X. (2018), DTU - Drone inspection images of wind turbine,
Mendeley Data V2, doi:10.17632/hd96prn3nc.2.
""".strip()
(OUT/"artifacts/CITATIONS.txt").write_text(CITATIONS)

print("\n=== Artifacts ===")
for p in [
    BEST_CKPT,
    OUT/"artifacts/summary_metrics.csv",
    OUT/"artifacts/per_class_ap.csv",
    OUT/"artifacts/detections_test.csv",
    OUT/"artifacts/executive_summary.csv",
    OUT/"artifacts/inference_fps.csv",
]:
    print(f"  {p}")